<a href="https://colab.research.google.com/github/nhahub/NHA-5-153/blob/main/Milestone3_Model_Development.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Milestone 3: Machine Learning Model Development and Optimization

## Task 1: Model Selection

### 1. Objective
The objective of this task is to select suitable machine learning classification algorithms to predict employee attrition.

The target variable is `Attrition`, where:
- 0 represents employees who stay.
- 1 represents employees who leave.

### 2. Class Imbalance Analysis
The dataset contains 49,000 employees:
- 36,470 employees (74.43%) belong to the non-attrition class.
- 12,530 employees (25.57%) belong to the attrition class.

This indicates a class imbalance problem. Therefore, models and training techniques that consider class imbalance will be evaluated.

### 3. Selected Models

Four classification algorithms were selected:

1. **Logistic Regression:** A simple baseline classification model that estimates the probability of employee attrition.
2. **Random Forest:** An ensemble learning algorithm that combines multiple decision trees to improve prediction performance.
3. **Gradient Boosting:** An ensemble algorithm that builds models sequentially to correct previous prediction errors.
4. **XGBoost:** An optimized gradient boosting algorithm known for its performance on structured datasets.

### 4. Selection Strategy
The selected models will be trained and compared using appropriate evaluation metrics, including Precision, Recall, F1-score, and ROC-AUC.

Class imbalance handling techniques will also be considered to improve the detection of employees likely to leave the organization.


# Task 2 & 3: Model **Training** & Model **Evaluatio**

In [3]:



import pandas as pd

url = "https://raw.githubusercontent.com/nhahub/NHA-5-153/refs/heads/main/transformed_hr_dataset.csv"

df = pd.read_csv(url)

print(df.shape)
df.head()


(49000, 26)


,EmployeeID,Age,TenureYears,PerformanceRating,MonthlySalary,Attrition,SalaryToTenureRatio,Gender_Female,Gender_Male,Gender_Non-Binary,...,TenureCategory_Medium-term,TenureCategory_Long-term,SalaryBand_Low,SalaryBand_Medium,SalaryBand_High,AgeGroup_Young,AgeGroup_Early Career,AgeGroup_Mid Career,AgeGroup_Experienced,AgeGroup_Senior
0,23056,33.0,1.0,1.0,9734.51,0,4867.255000,1,0,0,...,0,0,0,0,1,0,1,0,0,0
1,47703,33.0,5.0,4.0,7552.38,0,1258.730000,1,0,0,...,1,0,0,0,1,0,1,0,0,0
2,7424,45.0,0.0,4.0,3931.63,0,3931.630000,0,1,0,...,0,0,1,0,0,0,0,0,1,0
3,34697,35.0,2.0,3.0,7992.26,0,2664.086667,0,0,1,...,0,0,0,0,1,0,0,1,0,0
4,31490,42.0,1.0,3.0,5215.60,1,2607.800000,0,1,0,...,0,0,0,1,0,0,0,1,0,0


In [4]:

# Remove EmployeeID because it is only an identifier
X = df.drop(columns=['EmployeeID', 'Attrition'])

# Target variable
y = df['Attrition']

print("Features shape:", X.shape)
print("Target shape:", y.shape)


Features shape: (49000, 24)
Target shape: (49000,)


In [5]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Set:", X_train.shape)
print("Testing Set:", X_test.shape)

print("\nTraining Target Distribution:")
print(y_train.value_counts(normalize=True))


Training Set: (39200, 24)
Testing Set: (9800, 24)

Training Target Distribution:
Attrition
0    0.744286
1    0.255714
Name: proportion, dtype: float64


In [6]:

!pip install -q imbalanced-learn xgboost


In [7]:

from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_resampled, y_train_resampled = smote.fit_resample(
    X_train,
    y_train
)

print("Before SMOTE:")
print(y_train.value_counts())

print("\nAfter SMOTE:")
print(y_train_resampled.value_counts())


Before SMOTE:
Attrition
0    29176
1    10024
Name: count, dtype: int64

After SMOTE:
Attrition
1    29176
0    29176
Name: count, dtype: int64


In [8]:

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier

from xgboost import XGBClassifier
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

import pandas as pd
import numpy as np


In [9]:

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


In [14]:

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100,
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )
}


In [15]:

pipelines = {}

for name, model in models.items():

    steps = []

    if name == "Logistic Regression":
        steps.append(("scaler", StandardScaler()))

    steps.append(("smote", SMOTE(random_state=42)))
    steps.append(("model", model))

    pipelines[name] = Pipeline(steps)



In [16]:

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv_results = []

for name, pipeline in pipelines.items():

    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    cv_results.append({
        "Model": name,
        "Accuracy": scores["test_accuracy"].mean(),
        "Precision": scores["test_precision"].mean(),
        "Recall": scores["test_recall"].mean(),
        "F1-score": scores["test_f1"].mean(),
        "ROC-AUC": scores["test_roc_auc"].mean()
    })

results_df = pd.DataFrame(cv_results)

results_df


,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,Logistic Regression,0.514541,0.294598,0.644352,0.404331,0.579724
1,Random Forest,0.688342,0.274164,0.132782,0.178911,0.519520
2,Gradient Boosting,0.744031,0.217172,0.000898,0.001788,0.571633
3,XGBoost,0.743878,0.342172,0.001796,0.003572,0.566800


In [17]:

for name, pipeline in pipelines.items():
    print(f"\n{name}")
    print(pipeline)



Logistic Regression
Pipeline(steps=[('scaler', StandardScaler()), ('smote', SMOTE(random_state=42)),
                ('model', LogisticRegression(max_iter=1000, random_state=42))])

Random Forest
Pipeline(steps=[('smote', SMOTE(random_state=42)),
                ('model', RandomForestClassifier(n_jobs=-1, random_state=42))])

Gradient Boosting
Pipeline(steps=[('smote', SMOTE(random_state=42)),
                ('model', GradientBoostingClassifier(random_state=42))])

XGBoost
Pipeline(steps=[('smote', SMOTE(random_state=42)),
                ('model',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsample_bynode=None,
                               colsample_bytree=None, device=None,
                               early_stopping_rounds=None,
                               enable_categorical=True, eval_metric='logloss',
                               feature_types=None, feature_weights=None,
         

In [18]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

test_results = []

for name, pipeline in pipelines.items():

    # Train the model
    pipeline.fit(X_train, y_train)

    # Make predictions
    y_pred = pipeline.predict(X_test)

    # Get prediction probabilities
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    # Evaluate the model
    test_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

test_results_df = pd.DataFrame(test_results)

test_results_df


,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,Logistic Regression,0.504796,0.288673,0.639665,0.397816,0.575703
1,Random Forest,0.687857,0.273175,0.132881,0.178792,0.522828
2,Gradient Boosting,0.744286,0.000000,0.000000,0.000000,0.566687
3,XGBoost,0.743980,0.363636,0.001596,0.003178,0.565010


In [19]:

for name, pipeline in pipelines.items():

    y_pred = pipeline.predict(X_test)

    print("=" * 50)
    print(name)
    print("=" * 50)

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))


Logistic Regression

Confusion Matrix:
[[3344 3950]
 [ 903 1603]]

Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.46      0.58      7294
           1       0.29      0.64      0.40      2506

    accuracy                           0.50      9800
   macro avg       0.54      0.55      0.49      9800
weighted avg       0.66      0.50      0.53      9800

Random Forest

Confusion Matrix:
[[6408  886]
 [2173  333]]

Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.88      0.81      7294
           1       0.27      0.13      0.18      2506

    accuracy                           0.69      9800
   macro avg       0.51      0.51      0.49      9800
weighted avg       0.63      0.69      0.65      9800

Gradient Boosting

Confusion Matrix:
[[7294    0]
 [2506    0]]

Classification Report:
              precision    recall  f1-score   support

           0       0.74      

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# **Task** 4 : **Hyperparameter Tuning**

In [20]:

from sklearn.model_selection import GridSearchCV

# Select the best model so far
logistic_pipeline = pipelines["Logistic Regression"]

# Define hyperparameter values
param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100]
}

# Apply Grid Search
grid_search = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

# Train and find the best parameters
grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best Cross-Validation F1-score:", grid_search.best_score_)


Fitting 5 folds for each of 5 candidates, totalling 25 fits
Best Parameters: {'model__C': 10}
Best Cross-Validation F1-score: 0.40438073798437524


In [21]:

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score
)

# Get the best model
best_model = grid_search.best_estimator_

# Predict on test data
y_pred_tuned = best_model.predict(X_test)
y_prob_tuned = best_model.predict_proba(X_test)[:, 1]

# Display evaluation results
print("Tuned Logistic Regression Results")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_tuned))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_tuned,
    zero_division=0
))

print("ROC-AUC:", roc_auc_score(y_test, y_prob_tuned))


Tuned Logistic Regression Results

Confusion Matrix:
[[3344 3950]
 [ 903 1603]]

Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.46      0.58      7294
           1       0.29      0.64      0.40      2506

    accuracy                           0.50      9800
   macro avg       0.54      0.55      0.49      9800
weighted avg       0.66      0.50      0.53      9800

ROC-AUC: 0.5757045170012589


In [22]:

from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_recall_curve
import numpy as np

# Get out-of-fold probabilities using training data
train_probabilities = cross_val_predict(
    best_model,
    X_train,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

# Calculate precision and recall
precision_values, recall_values, thresholds = precision_recall_curve(
    y_train,
    train_probabilities
)

# Calculate F1-score for each threshold
f1_values = (
    2 * precision_values[:-1] * recall_values[:-1]
    / (precision_values[:-1] + recall_values[:-1] + 1e-10)
)

# Find threshold with highest F1-score
best_index = np.argmax(f1_values)
best_threshold = thresholds[best_index]

print("Best Threshold:", best_threshold)
print("Training F1-score:", f1_values[best_index])


Best Threshold: 0.43933350675538174
Training F1-score: 0.42308526791735174


In [23]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Predict using the selected threshold
y_pred_threshold = (
    y_prob_tuned >= best_threshold
).astype(int)

print("Threshold:", best_threshold)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_threshold))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_threshold,
    zero_division=0
))

print("Accuracy:", accuracy_score(y_test, y_pred_threshold))
print("Precision:", precision_score(y_test, y_pred_threshold, zero_division=0))
print("Recall:", recall_score(y_test, y_pred_threshold, zero_division=0))
print("F1-score:", f1_score(y_test, y_pred_threshold, zero_division=0))


Threshold: 0.43933350675538174

Confusion Matrix:
[[1588 5706]
 [ 313 2193]]

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.22      0.35      7294
           1       0.28      0.88      0.42      2506

    accuracy                           0.39      9800
   macro avg       0.56      0.55      0.38      9800
weighted avg       0.69      0.39      0.36      9800

Accuracy: 0.38581632653061226
Precision: 0.27763007975693127
Recall: 0.8750997605746209
F1-score: 0.421528111484863


In [24]:

final_comparison = pd.DataFrame([
    {
        "Model": "Logistic Regression (Default Threshold)",
        "Accuracy": 0.504796,
        "Precision": 0.288673,
        "Recall": 0.639665,
        "F1-score": 0.402,
        "ROC-AUC": 0.575703
    },
    {
        "Model": "Logistic Regression (Tuned Threshold)",
        "Accuracy": accuracy_score(y_test, y_pred_threshold),
        "Precision": precision_score(y_test, y_pred_threshold),
        "Recall": recall_score(y_test, y_pred_threshold),
        "F1-score": f1_score(y_test, y_pred_threshold),
        "ROC-AUC": roc_auc_score(y_test, y_prob_tuned)
    }
])

final_comparison


,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,Logistic Regression (Default Threshold),0.504796,0.288673,0.639665,0.402000,0.575703
1,Logistic Regression (Tuned Threshold),0.385816,0.277630,0.875100,0.421528,0.575705


# **Model Evaluation Report** :


# Milestone 3: Machine Learning Model Development and Optimization

## 1. Introduction

The main objective of this milestone is to develop and optimize machine learning classification models to predict employee attrition.

Employee attrition refers to employees leaving an organization. Predicting employee attrition can help companies identify employees who may leave and take preventive actions to improve employee retention.

In this milestone, four machine learning algorithms were selected, trained, and evaluated:

1. Logistic Regression
2. Random Forest
3. Gradient Boosting
4. XGBoost

The models were evaluated using different performance metrics, including Accuracy, Precision, Recall, F1-score, and ROC-AUC.

The project also applies SMOTE to handle class imbalance and GridSearchCV to optimize model hyperparameters.



## 2. Model Selection

Four classification algorithms were selected for the employee attrition prediction task.

### 2.1 Logistic Regression

Logistic Regression is a classification algorithm used to predict the probability of an observation belonging to a specific class.

It was selected because it is simple, computationally efficient, and provides a useful baseline for binary classification problems.

### 2.2 Random Forest

Random Forest is an ensemble learning algorithm that combines multiple Decision Trees to improve prediction performance.

It was selected because it can capture nonlinear relationships and handle different types of features.

### 2.3 Gradient Boosting

Gradient Boosting builds multiple weak prediction models sequentially, where each new model attempts to correct the errors made by previous models.

It was selected because it can capture complex patterns in structured datasets.

### 2.4 XGBoost

XGBoost is an optimized gradient boosting algorithm designed to provide efficient and powerful predictions.

It was selected because it offers regularization techniques and supports complex classification tasks.

These four models provide different learning approaches, allowing us to compare their performance and identify the most suitable model.



## 3. Data Preparation and Train-Test Split

The transformed dataset was used for model development.

The dataset contains 49,000 records and 26 columns.

Before training, the EmployeeID column was excluded because it is an identifier and does not provide meaningful predictive information.

The Attrition column was used as the target variable, while the remaining columns were used as input features.

Therefore:

- Total records: 49,000
- Total columns: 26
- Input features: 24
- Target variable: Attrition

The dataset was divided into two subsets:

- Training set: 80% (39,200 records)
- Testing set: 20% (9,800 records)

The training set was used to train the models, while the testing set was kept separate to evaluate their performance on unseen data.



## 4. Handling Class Imbalance

The target variable contains an imbalanced distribution:

- Class 0 (No Attrition): 36,470 records (74.43%)
- Class 1 (Attrition): 12,530 records (25.57%)

This imbalance may cause machine learning models to favor the majority class and perform poorly when identifying employees who leave the organization.

To address this issue, the Synthetic Minority Over-sampling Technique (SMOTE) was applied.

SMOTE generates synthetic samples for the minority class to balance the training data.

It was applied inside the machine learning pipelines to ensure that oversampling occurs only during the training process of each cross-validation fold.

This approach helps prevent data leakage from synthetic samples into validation folds.

The test set was not modified using SMOTE.



## 5. Model Training and Cross Validation

The four selected models were trained using the training dataset.

A Pipeline was created for each model to organize preprocessing and training steps.

For Logistic Regression, StandardScaler was applied before SMOTE because Logistic Regression is sensitive to differences in feature scales.

SMOTE was then applied to balance the training data before model fitting.

For the tree-based models, SMOTE was applied before training the models.

### Cross Validation

Stratified 5-Fold Cross Validation was used to evaluate model performance during training.

The training data was divided into five folds. Each fold was used once for validation, while the remaining four folds were used for training.

Stratification helps preserve the original class distribution in each fold.

The following metrics were calculated:

- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC

Cross-validation provides a more reliable estimate of model performance than relying on a single training-validation split.



### 5.1 Cross Validation Results

The following table presents the average performance of the four models across five cross-validation folds.

| Model | Accuracy | Precision | Recall | F1-score | ROC-AUC |
|---|---:|---:|---:|---:|---:|
| Logistic Regression | 0.5145 | 0.2946 | 0.6444 | 0.4043 | 0.5797 |
| Random Forest | 0.6883 | 0.2742 | 0.1328 | 0.1789 | 0.5195 |
| Gradient Boosting | 0.7440 | 0.2172 | 0.0009 | 0.0018 | 0.5716 |
| XGBoost | 0.7439 | 0.3422 | 0.0018 | 0.0036 | 0.5668 |

### Interpretation

Logistic Regression achieved the highest Recall and F1-score among the four models.

Random Forest achieved a higher Accuracy but had a low Recall.

Gradient Boosting and XGBoost achieved relatively high Accuracy but failed to identify most positive Attrition cases.

Therefore, Accuracy alone is not sufficient for selecting the best model in this imbalanced classification problem.



## 6. Model Evaluation

After training and cross-validation, the models were evaluated using the independent testing dataset containing 9,800 records.

The following evaluation metrics were used:

### Accuracy
Measures the proportion of all correct predictions.

### Precision
Measures how many predicted positive cases were actually positive.

### Recall
Measures how many actual positive cases were correctly identified.

### F1-score
The harmonic mean of Precision and Recall. It provides a balance between these two metrics.

### ROC-AUC
Measures the model's ability to distinguish between positive and negative classes across different classification thresholds.

Confusion Matrices and Classification Reports were also generated to provide a detailed evaluation of each model.



### 6.1 Test Set Evaluation Results

| Model | Accuracy | Precision | Recall | F1-score | ROC-AUC |
|---|---:|---:|---:|---:|---:|
| Logistic Regression | 0.5048 | 0.2887 | 0.6397 | 0.3978 | 0.5757 |
| Random Forest | 0.6879 | 0.2732 | 0.1329 | 0.1789 | 0.5228 |
| Gradient Boosting | 0.7443 | 0.0000 | 0.0000 | 0.0000 | 0.5667 |
| XGBoost | 0.7440 | 0.3636 | 0.0016 | 0.0032 | 0.5650 |

### Interpretation

Logistic Regression achieved the best Recall and F1-score among the evaluated models.

Random Forest achieved higher Accuracy but identified only a small proportion of actual Attrition cases.

Gradient Boosting predicted all test records as Class 0, meaning it failed to identify any employees who left.

XGBoost identified only a very small number of positive cases.

These results indicate that Logistic Regression was the most suitable initial model for identifying employee attrition among the four tested algorithms.



### 6.2 Confusion Matrix Analysis

A Confusion Matrix was used to understand the types of correct and incorrect predictions made by each model.

The matrix contains four values:

- True Negative (TN): Correctly predicted employees who did not leave.
- False Positive (FP): Employees predicted to leave but actually did not.
- False Negative (FN): Employees who left but were not identified by the model.
- True Positive (TP): Correctly predicted employees who left.

#### Logistic Regression

Confusion Matrix:

| | Predicted 0 | Predicted 1 |
|---|---:|---:|
| Actual 0 | 3,344 | 3,950 |
| Actual 1 | 903 | 1,603 |

The model correctly identified 1,603 employees who left the organization.

However, it missed 903 actual Attrition cases and produced 3,950 false positive predictions.

#### Random Forest

Confusion Matrix:

| | Predicted 0 | Predicted 1 |
|---|---:|---:|
| Actual 0 | 6,408 | 886 |
| Actual 1 | 2,173 | 333 |

The model correctly identified only 333 employees who left, while missing 2,173 actual Attrition cases.

#### Gradient Boosting

Confusion Matrix:

| | Predicted 0 | Predicted 1 |
|---|---:|---:|
| Actual 0 | 7,294 | 0 |
| Actual 1 | 2,506 | 0 |

The model predicted every employee as Class 0 and failed to identify any Attrition cases.

#### XGBoost

Confusion Matrix:

| | Predicted 0 | Predicted 1 |
|---|---:|---:|
| Actual 0 | 7,287 | 7 |
| Actual 1 | 2,502 | 4 |

The model identified only four actual Attrition cases, which indicates very weak performance for the positive class.



## 7. Hyperparameter Tuning

Hyperparameter tuning is the process of finding the best model settings to improve prediction performance.

After comparing the four models, Logistic Regression was selected for further optimization because it achieved the highest Recall and F1-score.

GridSearchCV was used to test different values of the regularization parameter C.

The tested values were:

C = [0.01, 0.1, 1, 10, 100]

The F1-score was selected as the optimization metric because it balances Precision and Recall.

Five-fold cross-validation was used during the search.

### 7.1 GridSearchCV Results

The best parameter was:

- Best C: 10
- Best Cross-Validation F1-score: 0.4044

The optimized Logistic Regression model was then evaluated on the independent test set.

### 7.2 Tuning Interpretation

The tuned model produced almost identical test performance to the original Logistic Regression model.

This indicates that changing the regularization parameter alone did not significantly improve the model's performance.



## 8. Threshold Optimization

Logistic Regression uses a default classification threshold of 0.5.

If the predicted probability of Attrition is greater than or equal to 0.5, the employee is classified as Class 1. Otherwise, the employee is classified as Class 0.

To improve the detection of employees who may leave, threshold optimization was performed.

Out-of-fold prediction probabilities were generated using five-fold cross-validation on the training data.

The Precision-Recall curve was then used to identify the threshold that achieved the highest F1-score.

The selected threshold was approximately:

0.4393

This threshold was applied to the test probabilities of the tuned Logistic Regression model.



### 8.1 Threshold Optimization Results

The optimized threshold produced the following test results:

| Metric | Default Threshold (0.5) | Tuned Threshold (0.4393) |
|---|---:|---:|
| Accuracy | 0.5048 | 0.3858 |
| Precision | 0.2887 | 0.2776 |
| Recall | 0.6397 | 0.8751 |
| F1-score | 0.3978 | 0.4215 |
| ROC-AUC | 0.5757 | 0.5757 |

### 8.2 Confusion Matrix After Threshold Optimization

| | Predicted 0 | Predicted 1 |
|---|---:|---:|
| Actual 0 | 1,588 | 5,706 |
| Actual 1 | 313 | 2,193 |

### Interpretation

After reducing the classification threshold from 0.5 to 0.4393, the model became more sensitive to positive Attrition cases.

Recall increased from approximately 63.97% to 87.51%.

F1-score also improved from approximately 39.78% to 42.15%.

However, Accuracy decreased to 38.58%, and Precision decreased slightly.

This happened because the model classified more employees as potential Attrition cases, increasing the number of False Positives.

The ROC-AUC remained almost unchanged because threshold adjustment changes the final class predictions but does not change the model's predicted probabilities.

Therefore, threshold optimization improved Attrition detection, but it also increased false alarms.



## 9. Model Comparison and Final Model Selection

The four classification models were compared based on their performance on the testing dataset.

Logistic Regression achieved the best Recall and F1-score among the four original models.

Further optimization was performed using GridSearchCV and threshold adjustment.

The final comparison between the two Logistic Regression configurations is shown below:

| Model | Accuracy | Precision | Recall | F1-score | ROC-AUC |
|---|---:|---:|---:|---:|---:|
| Logistic Regression (Default Threshold) | 0.5048 | 0.2887 | 0.6397 | 0.3978 | 0.5757 |
| Logistic Regression (Tuned Threshold) | 0.3858 | 0.2776 | 0.8751 | 0.4215 | 0.5757 |

### Final Model Selection

The tuned Logistic Regression model with a threshold of 0.4393 was selected as the preferred configuration when the main objective is to identify as many employees at risk of leaving as possible.

It achieved:

- Recall: 87.51%
- Precision: 27.76%
- F1-score: 42.15%
- ROC-AUC: 57.57%

However, this model produces a high number of False Positive predictions.

If reducing false alarms and maintaining higher Accuracy are more important, the default threshold configuration may be more appropriate.

The final choice therefore depends on the organization's priorities and the relative cost of missed Attrition cases versus false alarms.

The overall model performance is still limited, and further improvements are needed before using the model in a real business environment.



## 10. Conclusion

In this milestone, four machine learning classification algorithms were developed and evaluated to predict employee attrition.

SMOTE was applied to address the imbalance in the target variable, and five-fold Stratified Cross Validation was used to compare model performance.

Among the four models, Logistic Regression achieved the best initial Recall and F1-score.

GridSearchCV was then applied to optimize its regularization parameter, resulting in a best C value of 10.

Although hyperparameter tuning did not produce a significant improvement on the test set, threshold optimization improved the model's ability to identify employees who leave.

The final Logistic Regression configuration achieved a Recall of 87.51% and an F1-score of 42.15%, but its Accuracy decreased to 38.58% due to an increase in false positive predictions.

Overall, Logistic Regression was the most suitable model among the tested configurations for prioritizing Attrition detection. However, its low ROC-AUC indicates that the model still has limited discrimination ability.

Further work is required to improve predictive performance, investigate additional features, and explore alternative modeling techniques before considering real-world deployment.
